In [1]:
import os
import shutil
import random

# **Train & Valid Split**

In [2]:
SOURCE_DIR = "paddy_disease_classification/train_images"
TRAIN_DIR = "dataset/train"
VAL_DIR = "dataset/val"

SPLIT_RATIO = 0.8

for folder in [TRAIN_DIR, VAL_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)

for class_name in os.listdir(SOURCE_DIR):
    class_path = os.path.join(SOURCE_DIR, class_name)

    if not os.path.isdir(class_path):
        continue

    images = os.listdir(class_path)
    random.shuffle(images)

    split_index = int(len(images) * SPLIT_RATIO)

    train_images = images[:split_index]
    val_images = images[split_index:]

    os.makedirs(os.path.join(TRAIN_DIR, class_name), exist_ok=True)
    os.makedirs(os.path.join(VAL_DIR, class_name), exist_ok=True)

    for img in train_images:
        shutil.copy(
            os.path.join(class_path, img),
            os.path.join(TRAIN_DIR, class_name, img)
        )

    for img in val_images:
        shutil.copy(
            os.path.join(class_path, img),
            os.path.join(VAL_DIR, class_name, img)
        )

print("✅ Dataset split completed!")

✅ Dataset split completed!


In [ ]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image

from dataprocessor import load_and_preprocess

# ===== CONFIG =====
IMG_SIZE = 224
DATA_DIR = "paddy_disease_classification/train_images"
CSV_PATH = "paddy_disease_classification/train.csv"

# ===== LOAD DATA =====
X, y_disease, y_variety, y_age, disease_encoder, variety_encoder, max_age = load_and_preprocess(DATA_DIR, CSV_PATH)

# ===== BASE MODEL =====
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)

# ===== OUTPUTS =====
disease_output = layers.Dense(len(disease_encoder.classes_), activation='softmax', name='disease')(x)

variety_output = layers.Dense(len(variety_encoder.classes_), activation='softmax', name='variety')(x)

age_output = layers.Dense(1, activation='linear', name='age')(x)


model = models.Model(
    inputs=base_model.input,
    outputs=[disease_output, variety_output, age_output]
)

model.compile(
    optimizer='adam',
    loss={
        'disease': 'sparse_categorical_crossentropy',
        'variety': 'sparse_categorical_crossentropy',
        'age': 'mse'
    },
    metrics={
        'disease': 'accuracy',
        'variety': 'accuracy',
        'age': 'mae'
    }
)

model.fit(
    X,
    {
        'disease': y_disease,
        'variety': y_variety,
        'age': y_age
    },
    epochs=10,
    batch_size=32
)

model.save("multi_output_rice_model.h5")

print("✅ Model trained and saved!")

def predict(img_path):
    model = tf.keras.models.load_model("multi_output_rice_model.h5")

    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img = image.img_to_array(img) / 255.0
    img = np.expand_dims(img, axis=0)

    pred_disease, pred_variety, pred_age = model.predict(img)

    disease = disease_encoder.inverse_transform([np.argmax(pred_disease)])
    variety = variety_encoder.inverse_transform([np.argmax(pred_variety)])
    age = pred_age[0][0] * max_age

    print("Disease:", disease[0])
    print("Variety:", variety[0])
    print("Age (approx):", int(age))
